# TalkMate: английская речь — замер распознавания и детектор звуков-пауз

Всё бесплатно, на Kaggle.

**Перед запуском**
1. Settings → Accelerator: **GPU T4 x1**. Settings → Internet: **On** (нужно подтвердить телефон в профиле Kaggle).
2. Add Input → Datasets → загрузите `talkmate-code.zip` (лежит в папке проекта `output/kaggle/`) как свой датасет `talkmate-code`.
3. Run All. Шаг 1 занимает около 20–40 минут, шаг 2 — несколько часов: звук 199 эпизодов скачивается по одному и сразу удаляется.

**Что получится в Output:** `en_asr_benchmark.json`, `filler_head.pt`, `filler_results.json`. Файл `filler_head.pt` положите в `backend/speech_analysis/weights/`.

Лицензии: AMI — CC BY 4.0; разметка PodcastFillers — некоммерческая (подходит для учебного проекта).

In [ ]:
import glob, os, shutil, socket, subprocess, sys, zipfile

# 1. Интернет. Без него не поставить библиотеки и не скачать модели и данные.
try:
    socket.create_connection(("pypi.org", 443), timeout=5).close()
except OSError:
    raise RuntimeError("Нет интернета. Справа: Session options (или Settings) → Internet → On. "
                     "Если переключателя нет, подтвердите телефон: kaggle.com → Settings → Phone verification. "
                     "Потом Restart session и Run All.") from None

# 2. Код TalkMate. Kaggle сам распаковывает загруженный .zip, поэтому ищем и архив, и распакованную папку.
work = "/kaggle/working/talkmate"
zips = glob.glob("/kaggle/input/**/talkmate-code.zip", recursive=True)
folders = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/backend/speech_analysis", recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(work)
elif folders:
    shutil.copytree(folders[0], work, dirs_exist_ok=True)
else:
    raise RuntimeError("Не найден код TalkMate. Справа: Add Input → Your Datasets → talkmate-code "
                     "(если датасета ещё нет: Create → New Dataset → загрузите talkmate-code.zip). "
                     "Сейчас в /kaggle/input: " + str(os.listdir("/kaggle/input")))

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "onnx-asr[cpu,hub]>=0.12", "jiwer", "num2words",
                "librosa==0.10.2.post1", "soundfile", "datasets"], check=True)
os.chdir(f"{work}/backend"); sys.path.insert(0, os.getcwd())
os.makedirs("/kaggle/working/out", exist_ok=True)
print("Готово:", os.getcwd())

## Шаг 1. Исходный уровень: что теряет текущая английская модель

AMI, гарнитурные микрофоны, тестовая часть: 600 реплик от 2 до 20 секунд, половина — с «um»/«uh».
Считаются доля ошибок в словах (WER), доля сохранённых звуков-пауз и словосочетаний «you know», «I mean», доля сохранённых повторов.

In [ ]:
!python -m training.en_asr_benchmark --limit 600 \
    --models nemo-parakeet-tdt-0.6b-v3:int8 nemo-parakeet-tdt-0.6b-v2 \
    --out /kaggle/working/out/en_asr_benchmark.json

## Шаг 2. Детектор звуков-пауз по звуку

Разметка PodcastFillers с Zenodo (22 МБ), звук эпизодов с Hugging Face. Признаки — замороженный WavLM модели эмоций
TalkMate, голова — та же `build_head`, что у эмоций, два класса. Порог выбирается на проверочной части так,
чтобы точность была не ниже 0,9: лишний упрёк в «э-э» хуже пропуска.

In [ ]:
!wget -q -O /kaggle/working/PodcastFillers.csv "https://zenodo.org/records/7121457/files/PodcastFillers.csv?download=1"
import pandas as pd
pf = pd.read_csv('/kaggle/working/PodcastFillers.csv')
print(pf.shape); print(pf.columns.tolist())
print(pf['episode_split_subset'].value_counts()); print(pf['consolidated_label'].value_counts())
print(pf['podcast_filename'].head(3).tolist())

In [ ]:
# Проверочная и тестовая части берутся целиком, обучающая — все звуки-паузы и по 1,5 других события на каждую.
# При ошибке ячейка останавливается и показывает причину: что лежит в разметке и в хранилище со звуком.
import subprocess, sys
for split in ['validation', 'test', 'train']:
    subprocess.run([sys.executable, '-m', 'training.fillers_detector', 'extract', '--csv', '/kaggle/working/PodcastFillers.csv',
                    '--split', split, '--out', f'/kaggle/working/feat_{split}.npz'], check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'training.fillers_detector', 'train', '--train', '/kaggle/working/feat_train.npz',
                '--val', '/kaggle/working/feat_validation.npz', '--out', '/kaggle/working/out/filler_head.pt',
                '--epochs', '30', '--min-precision', '0.9'], check=True)
subprocess.run([sys.executable, '-m', 'training.fillers_detector', 'evaluate', '--data', '/kaggle/working/feat_test.npz',
                '--head', '/kaggle/working/out/filler_head.pt', '--out', '/kaggle/working/out/filler_results.json'], check=True)

## Шаг 3 (по желанию). Переносится ли детектор на русское «э-э»

Реплики SOVA RuDevices, где в эталоне записан звук-пауза, и столько же реплик без него. Считаются полнота
(какую долю «э-э» детектор нашёл) и ложные срабатывания на репликах без пауз. Если Hugging Face не отдаёт набор,
шаг пропускается.

In [ ]:
import io, json, numpy as np, soundfile as sf
from speech_analysis.emotion_head import load_encoder, load_head, pick_device
from speech_analysis.fillers import find_fillers
from training.asr_benchmark import normalize_text, SOUNDS
from training.fillers_detector import DEFAULT_ENCODER, scan
try:
    from datasets import load_dataset
    try:
        sova = load_dataset('bond005/sova_rudevices', split='test')
    except Exception:
        sova = load_dataset('parquet', data_files={'test': 'hf://datasets/bond005/sova_rudevices@refs%2Fconvert%2Fparquet/default/test/*.parquet'}, split='test')
    model, extractor = load_encoder(DEFAULT_ENCODER)
    head, meta = load_head('/kaggle/working/out/filler_head.pt', pick_device())
    with_sound, without = [], []
    for row in sova:
        ref = normalize_text(row['transcription'], keep_hyphens=True)
        n = sum(1 for w in find_fillers(ref) if w in SOUNDS)
        (with_sound if n else without).append((row, n))
    without = without[:len(with_sound)]
    found = total = false_alarms = 0
    for group, is_sound in ((with_sound, True), (without, False)):
        for row, n in group:
            samples = np.asarray(row['audio']['array'], dtype=np.float32)
            events = scan(model, extractor, head, samples, meta['threshold'])
            if is_sound:
                total += n; found += min(n, len(events))
            else:
                false_alarms += bool(events)
    ru = {'utterances_with_sounds': len(with_sound), 'sound_recall': round(found / max(total, 1), 4),
          'false_alarm_utterances': round(false_alarms / max(len(without), 1), 4)}
except Exception as exc:
    ru = {'error': str(exc)}
print(ru)
json.dump(ru, open('/kaggle/working/out/filler_russian.json', 'w'), ensure_ascii=False, indent=2)

In [ ]:
!ls -la /kaggle/working/out && cat /kaggle/working/out/filler_results.json